# Semantic Linker V3 — Fine-tuning XLM-R Base sur Kaggle

Notebook complet pour entraîner le **Semantic Linker quantitatif V3** sur le dataset global stress.

### Dataset attendu
- `semantic_linker_train_v3_combined.jsonl` — **98 100** exemples
- `semantic_linker_val_v3_global_stress.jsonl` — **15 360** exemples
- `semantic_linker_test_v3_global_stress.jsonl` — **15 360** exemples
- optionnel : `semantic_linker_v3_QA_REPORT.json`
- optionnel : `semantic_linker_v3_metadata.json`

### Contrat conservé
Le notebook garde exactement l'interface du runtime local :
- encoder **XLM-R Base**
- deux heads : `FIELD` et `ROLE`
- marqueurs `[Q] ... [/Q]`
- décodage contraint `dimension → FIELD → ROLE`
- calibration `confidence + margin`
- export : `encoder/`, `tokenizer/`, `classifier_heads.pt`, `labels.json`, `compatibility.json`, `thresholds.json`, `training_config.json`

### Important
Le `TEST` reste fermé jusqu'à la fin. Les seuils sont choisis **uniquement sur VAL**.  
Les anciens cas `clents / clints / noeuds compute` sont utilisés **uniquement comme probes de régression après le TEST**, pas pour choisir le modèle ou les thresholds.

Pour les formulations `X or Y`, les deux quantités sont des exemples positifs indépendants pour le même field. La décision entre les deux valeurs appartient au **guard en aval**, pas au Semantic Linker.


## 1 — Environnement Kaggle et hyperparamètres

Sur Kaggle :
1. activer **GPU** ;
2. ajouter le dataset V3 dans *Input* ;
3. laisser Internet activé pour télécharger `FacebookAI/xlm-roberta-base`, ou remplacer `MODEL_SOURCE` par le chemin d'un modèle XLM-R ajouté à Kaggle Input.

Le dataset est beaucoup plus grand que V1. On utilise gradient accumulation, AMP et gradient checkpointing pour rester compatible avec un GPU Kaggle 16 GB.


In [ ]:
import os, sys, math, json, random, shutil, subprocess, importlib.util, gc, time
from pathlib import Path
from collections import Counter, defaultdict

# ---------------------------------------------------------
# Dependencies
# ---------------------------------------------------------
required_packages = {
    "transformers": "transformers",
    "sklearn": "scikit-learn",
    "sentencepiece": "sentencepiece",
    "safetensors": "safetensors",
    "tqdm": "tqdm",
}
missing = [
    pip_name
    for module_name, pip_name in required_packages.items()
    if importlib.util.find_spec(module_name) is None
]
if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
import transformers
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    ConfusionMatrixDisplay,
)
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda":
    raise RuntimeError("GPU non détecté. Active un accélérateur GPU dans Kaggle.")

print("GPU:", torch.cuda.get_device_name(0))
props = torch.cuda.get_device_properties(0)
print("GPU memory:", round(props.total_memory / 1024**3, 2), "GB")
print("CUDA runtime:", torch.version.cuda)

# ---------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------
SEED = 42

def reset_all_seeds(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

reset_all_seeds()

# ---------------------------------------------------------
# Model / data
# ---------------------------------------------------------
MODEL_SOURCE = "FacebookAI/xlm-roberta-base"

TRAIN_FILENAME = "semantic_linker_train_v3_combined.jsonl"
VAL_FILENAME = "semantic_linker_val_v3_global_stress.jsonl"
TEST_FILENAME = "semantic_linker_test_v3_global_stress.jsonl"
QA_FILENAME = "semantic_linker_v3_QA_REPORT.json"
METADATA_FILENAME = "semantic_linker_v3_metadata.json"

EXPECTED_TRAIN = 98100
EXPECTED_VAL = 15360
EXPECTED_TEST = 15360

# ---------------------------------------------------------
# Training configuration
# Safe default for Kaggle 16 GB GPUs.
# If OOM on a 16 GB GPU: set BATCH_SIZE=4 and GRAD_ACCUM_STEPS=8 to keep effective batch size = 32.
# ---------------------------------------------------------
BATCH_SIZE = 8
GRAD_ACCUM_STEPS = 4
EFFECTIVE_BATCH_SIZE = BATCH_SIZE * GRAD_ACCUM_STEPS

MAX_EPOCHS = 5
EARLY_STOPPING_PATIENCE = 2

ENCODER_LR = 1.5e-5
HEAD_LR = 5e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
MAX_GRAD_NORM = 1.0
DROPOUT = 0.10

FIELD_LOSS_WEIGHT = 1.0
ROLE_LOSS_WEIGHT = 1.0
USE_ROLE_CLASS_WEIGHTS = True

USE_AMP = True
USE_GRADIENT_CHECKPOINTING = True
NUM_WORKERS = 2

# Selective prediction calibration.
TARGET_ACCEPTED_PRECISION = 0.98
MIN_UNRESOLVED_RECALL = 0.995

# Context cap. MAX_LENGTH is selected later from the real V3 corpus,
# but never above this value.
MAX_LENGTH_CAP = 256
MIN_CONTEXT_AFTER_Q = 24

# ---------------------------------------------------------
# Output paths
# ---------------------------------------------------------
WORK_DIR = Path("/kaggle/working/semantic_linker_xlmr_v3")
BEST_DIR = WORK_DIR / "best_checkpoint"
EXPORT_DIR = WORK_DIR / "final_export"

BEST_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print("Device:", DEVICE)
print("Model source:", MODEL_SOURCE)
print("Effective batch size:", EFFECTIVE_BATCH_SIZE)
print("Max epochs:", MAX_EPOCHS)


## 2 — Charger TRAIN / VAL / TEST et vérifier les contrats

Cette cellule :
- retrouve automatiquement les fichiers dans `/kaggle/input`;
- vérifie les tailles attendues ;
- vérifie les `sample_id` et `group_id` entre splits ;
- vérifie qu'il y a exactement un `[Q]...[/Q]` par exemple ;
- lit le rapport QA V3 s'il est présent.

**Aucune statistique du TEST n'est utilisée pour choisir les hyperparamètres.** Les contrôles ci-dessous sont uniquement structurels.


In [ ]:
def find_input_file(filename, required=True):
    matches = list(Path("/kaggle/input").rglob(filename))
    if not matches:
        if required:
            raise FileNotFoundError(f"{filename} introuvable sous /kaggle/input")
        return None
    if len(matches) > 1:
        print("WARNING: plusieurs copies trouvées pour", filename)
        for p in matches:
            print(" -", p)
    return matches[0]

def load_jsonl(path):
    records = []
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise RuntimeError(f"JSON invalide {path}:{line_no}: {e}") from e
    return records

TRAIN_PATH = find_input_file(TRAIN_FILENAME)
VAL_PATH = find_input_file(VAL_FILENAME)
TEST_PATH = find_input_file(TEST_FILENAME)
QA_PATH = find_input_file(QA_FILENAME, required=False)
METADATA_PATH = find_input_file(METADATA_FILENAME, required=False)

print("TRAIN:", TRAIN_PATH)
print("VAL  :", VAL_PATH)
print("TEST :", TEST_PATH)
print("QA   :", QA_PATH)
print("META :", METADATA_PATH)

train_records = load_jsonl(TRAIN_PATH)
val_records = load_jsonl(VAL_PATH)
test_records = load_jsonl(TEST_PATH)

assert len(train_records) == EXPECTED_TRAIN, (len(train_records), EXPECTED_TRAIN)
assert len(val_records) == EXPECTED_VAL, (len(val_records), EXPECTED_VAL)
assert len(test_records) == EXPECTED_TEST, (len(test_records), EXPECTED_TEST)

datasets = {
    "train": train_records,
    "val": val_records,
    "test": test_records,
}

REQUIRED_KEYS = {
    "sample_id", "group_id", "marked_text", "raw_quantity",
    "field", "role", "dimension", "language", "source"
}

for split, records in datasets.items():
    for i, r in enumerate(records):
        missing_keys = REQUIRED_KEYS - set(r)
        assert not missing_keys, f"{split}[{i}] missing {missing_keys}"
        assert r["marked_text"].count("[Q]") == 1
        assert r["marked_text"].count("[/Q]") == 1
        assert r["marked_text"].index("[Q]") < r["marked_text"].index("[/Q]")

sample_sets = {k: {r["sample_id"] for r in v} for k, v in datasets.items()}
group_sets = {k: {r["group_id"] for r in v} for k, v in datasets.items()}
text_sets = {k: {r["marked_text"] for r in v} for k, v in datasets.items()}

for split, records in datasets.items():
    assert len(sample_sets[split]) == len(records), f"duplicate sample_id in {split}"
    assert len(text_sets[split]) == len(records), f"duplicate marked_text in {split}"

for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    assert not (sample_sets[a] & sample_sets[b]), f"sample leakage {a}/{b}"
    assert not (group_sets[a] & group_sets[b]), f"group leakage {a}/{b}"
    assert not (text_sets[a] & text_sets[b]), f"marked_text leakage {a}/{b}"

print("✅ Sizes:", {k: len(v) for k, v in datasets.items()})
print("✅ Aucun sample_id leakage")
print("✅ Aucun group_id leakage")
print("✅ Aucun marked_text leakage")

for split, records in datasets.items():
    print(
        split.upper(),
        "n=", len(records),
        "languages=", dict(Counter(r["language"] for r in records)),
        "groups=", len({r["group_id"] for r in records}),
    )

if QA_PATH is not None:
    qa = json.loads(QA_PATH.read_text(encoding="utf-8"))
    print("\nQA status:", qa.get("status"))
    failed = [k for k, v in qa.get("checks", {}).items() if not v.get("pass")]
    if failed:
        raise RuntimeError(f"QA report contains failed checks: {failed}")
    print("✅ QA report: all checks PASS")

if METADATA_PATH is not None:
    v3_metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8"))
    print("✅ V3 metadata loaded")


## 3 — Distribution V3 : fields, rôles, langues et familles de stress

Le TRAIN est équilibré sur les 12 FIELD labels. Les rôles ne sont pas artificiellement uniformes, car toutes les combinaisons `FIELD × ROLE` ne sont pas valides.

On contrôle aussi les **16 familles de stress** :
- clean natural ;
- paraphrase/synonym ;
- semantic anchor typo ;
- mixed code-switch ;
- noisy mixed typos ;
- HPC hard negative ;
- multi-quantity collision ;
- alternative pair (`X or Y`) ;
- complex role/temporal ;
- current-vs-target ;
- long context ;
- short/telegraphic ;
- punctuation/case noise ;
- quantity words ;
- unit surface variation ;
- conversational/human-like.


In [ ]:
def stress_family(record):
    source = record.get("source", "")
    return source.split("+")[-1] if "+" in source else source

def distribution_frame(records, key):
    c = Counter(r[key] for r in records)
    return pd.DataFrame(
        sorted(c.items(), key=lambda x: (-x[1], x[0])),
        columns=[key, "count"]
    )

display(distribution_frame(train_records, "field"))
display(distribution_frame(train_records, "role"))
display(distribution_frame(train_records, "language"))

stress_counts = Counter(stress_family(r) for r in train_records)
display(
    pd.DataFrame(
        sorted(stress_counts.items(), key=lambda x: (-x[1], x[0])),
        columns=["stress_family", "count"]
    )
)

# Field balance is a deliberate V3 contract.
field_counts = Counter(r["field"] for r in train_records)
assert len(field_counts) == 12
assert len(set(field_counts.values())) == 1, field_counts
print("✅ TRAIN perfectly balanced by FIELD:", next(iter(field_counts.values())), "per field")


## 4 — Taxonomie et compatibility rules

Ces labels doivent rester identiques à ceux du runtime local.  
Le Transformer apprend les deux heads, puis le décodage élimine les couples physiquement/sémantiquement impossibles.


In [ ]:
FIELD_LABELS = [
    "requested_usable_capacity_tib",
    "client_count",
    "average_file_size_gb",
    "max_file_size_gb",
    "total_file_count",
    "read_write_ratio",
    "target_read_gbps",
    "target_write_gbps",
    "max_budget_usd",
    "max_power_w",
    "annual_growth_percent",
    "__UNRESOLVED__",
]

ROLE_LABELS = [
    "maximum_limit",
    "minimum_limit",
    "target",
    "current_value",
    "expected_value",
    "average_value",
    "total_count",
    "ratio_component",
    "growth_rate",
    "unspecified",
]

FIELD_TO_ID = {x: i for i, x in enumerate(FIELD_LABELS)}
ID_TO_FIELD = {i: x for x, i in FIELD_TO_ID.items()}
ROLE_TO_ID = {x: i for i, x in enumerate(ROLE_LABELS)}
ID_TO_ROLE = {i: x for x, i in ROLE_TO_ID.items()}

NUM_FIELDS = len(FIELD_LABELS)
NUM_ROLES = len(ROLE_LABELS)

UNRESOLVED_FIELD_ID = FIELD_TO_ID["__UNRESOLVED__"]
UNSPECIFIED_ROLE_ID = ROLE_TO_ID["unspecified"]

ALLOWED_FIELDS_BY_DIMENSION = {
    "capacity": {"requested_usable_capacity_tib", "__UNRESOLVED__"},
    "file_size": {"average_file_size_gb", "max_file_size_gb", "__UNRESOLVED__"},
    "throughput": {"target_read_gbps", "target_write_gbps", "__UNRESOLVED__"},
    "power": {"max_power_w", "__UNRESOLVED__"},
    "money": {"max_budget_usd", "__UNRESOLVED__"},
    "percent": {"read_write_ratio", "annual_growth_percent", "__UNRESOLVED__"},
    "count": {"client_count", "total_file_count", "__UNRESOLVED__"},
    "unknown": set(FIELD_LABELS),
}

ALLOWED_ROLES_BY_FIELD = {
    "requested_usable_capacity_tib": {"target", "current_value", "unspecified"},
    "client_count": {"total_count", "unspecified"},
    "average_file_size_gb": {"average_value", "unspecified"},
    "max_file_size_gb": {"maximum_limit", "unspecified"},
    "total_file_count": {"total_count", "unspecified"},
    "read_write_ratio": {"ratio_component", "unspecified"},
    "target_read_gbps": {"target", "current_value", "minimum_limit", "unspecified"},
    "target_write_gbps": {"target", "current_value", "minimum_limit", "unspecified"},
    "max_budget_usd": {"maximum_limit", "current_value", "unspecified"},
    "max_power_w": {"maximum_limit", "current_value", "expected_value", "unspecified"},
    "annual_growth_percent": {"growth_rate", "unspecified"},
    "__UNRESOLVED__": {"unspecified"},
}

assert NUM_FIELDS == 12 and NUM_ROLES == 10

for split, records in datasets.items():
    for r in records:
        assert r["field"] in FIELD_TO_ID
        assert r["role"] in ROLE_TO_ID
        assert r["dimension"] in ALLOWED_FIELDS_BY_DIMENSION
        assert r["field"] in ALLOWED_FIELDS_BY_DIMENSION[r["dimension"]]
        assert r["role"] in ALLOWED_ROLES_BY_FIELD[r["field"]]

print("✅ Taxonomy and compatibility contracts validated on TRAIN / VAL / TEST")


## 5 — Tokenizer XLM-R + `[Q]` et `[/Q]`

Les deux marqueurs sont ajoutés comme **special tokens atomiques**.  
Le même tokenizer exporté sera ensuite chargé par le runtime local.


In [ ]:
SPECIAL_TOKENS = {
    "additional_special_tokens": ["[Q]", "[/Q]"]
}

local_model = Path(MODEL_SOURCE).exists()

try:
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_SOURCE,
        use_fast=True,
        local_files_only=local_model,
    )
except Exception as e:
    raise RuntimeError(
        "Impossible de charger XLM-R. "
        "Active Internet dans Kaggle ou ajoute le modèle comme Kaggle Input "
        "et remplace MODEL_SOURCE par son chemin local."
    ) from e

vocab_before = len(tokenizer)
num_added = tokenizer.add_special_tokens(SPECIAL_TOKENS)

q_open_id = tokenizer.convert_tokens_to_ids("[Q]")
q_close_id = tokenizer.convert_tokens_to_ids("[/Q]")

assert q_open_id != tokenizer.unk_token_id
assert q_close_id != tokenizer.unk_token_id

print("Vocabulary before:", vocab_before)
print("Added special tokens:", num_added)
print("Vocabulary after:", len(tokenizer))
print("[Q] id:", q_open_id, "[/Q] id:", q_close_id)


## 6 — Audit des longueurs et choix automatique de `MAX_LENGTH`

V3 contient du contexte long et du code-switching. On ne fixe donc pas arbitrairement `96`.

L'audit parcourt les trois splits et mesure :
- longueur totale ;
- position de `[/Q]` ;
- besoin minimal pour conserver la quantité ciblée **et un peu de contexte après elle**.

Le TEST est utilisé ici uniquement pour une propriété **structurelle de tokenisation**, jamais pour sélectionner un paramètre de classification.


In [ ]:
def audit_lengths(records, batch_size=512, desc="token audit"):
    lengths = []
    q_close_positions = []

    for start in tqdm(range(0, len(records), batch_size), desc=desc):
        texts = [r["marked_text"] for r in records[start:start+batch_size]]
        enc = tokenizer(
            texts,
            add_special_tokens=True,
            truncation=False,
            padding=False,
        )
        for ids in enc["input_ids"]:
            lengths.append(len(ids))
            assert ids.count(q_open_id) == 1
            assert ids.count(q_close_id) == 1
            q_close_positions.append(ids.index(q_close_id))

    return np.asarray(lengths), np.asarray(q_close_positions)

all_lengths = []
all_qclose = []

for split, records in datasets.items():
    L, Q = audit_lengths(records, desc=f"Token audit {split}")
    all_lengths.append(L)
    all_qclose.append(Q)
    print(
        split,
        "p50=", int(np.percentile(L, 50)),
        "p95=", int(np.percentile(L, 95)),
        "p99=", int(np.percentile(L, 99)),
        "max=", int(L.max()),
        "max [/Q] pos=", int(Q.max()),
    )

lengths = np.concatenate(all_lengths)
qclose = np.concatenate(all_qclose)

def round_up_16(x):
    return int(math.ceil(x / 16) * 16)

# Preserve all target quantities and at least MIN_CONTEXT_AFTER_Q tokens
# after the closing marker whenever possible.
required_for_marker = int(qclose.max()) + 1 + MIN_CONTEXT_AFTER_Q
p995_length = int(np.percentile(lengths, 99.5))

MAX_LENGTH = min(
    MAX_LENGTH_CAP,
    round_up_16(max(required_for_marker, p995_length))
)

if int(qclose.max()) >= MAX_LENGTH - 2:
    raise RuntimeError(
        f"MAX_LENGTH={MAX_LENGTH} would truncate a [Q] target. "
        f"Increase MAX_LENGTH_CAP (current={MAX_LENGTH_CAP})."
    )

print("\nSelected MAX_LENGTH:", MAX_LENGTH)
print("p99.5 corpus length:", p995_length)
print("max [/Q] position:", int(qclose.max()))
print("✅ Every target marker survives truncation")


## 7 — Dataset PyTorch et DataLoaders

Le padding est dynamique par batch.  
Pour VAL/TEST on conserve aussi les métadonnées nécessaires aux analyses par langue, famille de stress et `alternative_pair`.


In [ ]:
class SemanticDataset(Dataset):
    def __init__(self, records):
        self.records = records

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r = self.records[idx]
        return {
            "text": r["marked_text"],
            "field_id": FIELD_TO_ID[r["field"]],
            "role_id": ROLE_TO_ID[r["role"]],
            "dimension": r["dimension"],
            "sample_id": r["sample_id"],
            "group_id": r["group_id"],
            "language": r["language"],
            "source": r["source"],
        }

def collate_batch(batch):
    enc = tokenizer(
        [x["text"] for x in batch],
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )
    enc["field_labels"] = torch.tensor(
        [x["field_id"] for x in batch], dtype=torch.long
    )
    enc["role_labels"] = torch.tensor(
        [x["role_id"] for x in batch], dtype=torch.long
    )
    enc["dimensions"] = [x["dimension"] for x in batch]
    enc["sample_ids"] = [x["sample_id"] for x in batch]
    enc["group_ids"] = [x["group_id"] for x in batch]
    enc["languages"] = [x["language"] for x in batch]
    enc["sources"] = [x["source"] for x in batch]
    return enc

generator = torch.Generator()
generator.manual_seed(SEED)

train_loader = DataLoader(
    SemanticDataset(train_records),
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=generator,
    collate_fn=collate_batch,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=(NUM_WORKERS > 0),
)

val_loader = DataLoader(
    SemanticDataset(val_records),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_batch,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=(NUM_WORKERS > 0),
)

test_loader = DataLoader(
    SemanticDataset(test_records),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_batch,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=(NUM_WORKERS > 0),
)

batch = next(iter(train_loader))
print("input_ids:", tuple(batch["input_ids"].shape))
print("FIELD labels:", tuple(batch["field_labels"].shape))
print("ROLE labels:", tuple(batch["role_labels"].shape))
print(tokenizer.decode(batch["input_ids"][0], skip_special_tokens=False)[:1000])


## 8 — XLM-R Base + deux heads

Architecture identique au runtime :
`XLM-R → <s> → Dropout → FIELD head + ROLE head`.

Le modèle est fine-tuné **end-to-end**. Les special-token embeddings sont appris avec l'encoder.


In [ ]:
class XLMRSemanticLinker(nn.Module):
    def __init__(
        self,
        model_source,
        tokenizer_size,
        num_fields,
        num_roles,
        dropout=0.1,
        gradient_checkpointing=True,
    ):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(
            model_source,
            local_files_only=Path(model_source).exists(),
        )
        self.encoder.resize_token_embeddings(tokenizer_size)

        if gradient_checkpointing:
            if hasattr(self.encoder.config, "use_cache"):
                self.encoder.config.use_cache = False
            self.encoder.gradient_checkpointing_enable()

        hidden = self.encoder.config.hidden_size

        self.dropout = nn.Dropout(dropout)
        self.field_head = nn.Linear(hidden, num_fields)
        self.role_head = nn.Linear(hidden, num_roles)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True,
        )
        h = self.dropout(out.last_hidden_state[:, 0, :])
        return self.field_head(h), self.role_head(h)

reset_all_seeds(SEED)

model = XLMRSemanticLinker(
    MODEL_SOURCE,
    len(tokenizer),
    NUM_FIELDS,
    NUM_ROLES,
    DROPOUT,
    USE_GRADIENT_CHECKPOINTING,
).to(DEVICE)

print("Hidden:", model.encoder.config.hidden_size)
print("Layers:", model.encoder.config.num_hidden_layers)
print("Heads:", model.encoder.config.num_attention_heads)
print("Total params:", f"{sum(p.numel() for p in model.parameters()):,}")
print("Trainable:", f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

model.eval()
with torch.no_grad():
    fl, rl = model(
        batch["input_ids"][:2].to(DEVICE),
        batch["attention_mask"][:2].to(DEVICE),
    )

assert fl.shape == (2, NUM_FIELDS)
assert rl.shape == (2, NUM_ROLES)
print("✅ Forward pass:", fl.shape, rl.shape)


## 9 — Décodage contraint FIELD + ROLE

On score uniquement les couples valides pour la dimension détectée en amont par `QuantityScanner`.

La confiance correspond à la probabilité normalisée du meilleur couple valide.  
La marge = `P(top1) - P(top2)`.


In [ ]:
def valid_pair_ids_for_dimension(dimension):
    pairs = []
    for field in FIELD_LABELS:
        if field not in ALLOWED_FIELDS_BY_DIMENSION[dimension]:
            continue
        fi = FIELD_TO_ID[field]
        for role in ROLE_LABELS:
            if role in ALLOWED_ROLES_BY_FIELD[field]:
                pairs.append((fi, ROLE_TO_ID[role]))
    return pairs

VALID_PAIR_CACHE = {
    d: valid_pair_ids_for_dimension(d)
    for d in ALLOWED_FIELDS_BY_DIMENSION
}

def decode_valid_pairs(field_logits, role_logits, dimensions):
    flp = torch.log_softmax(field_logits, dim=-1)
    rlp = torch.log_softmax(role_logits, dim=-1)

    pred_fields, pred_roles, confidence, margin = [], [], [], []

    for i, dim in enumerate(dimensions):
        pairs = VALID_PAIR_CACHE[dim]
        scores = torch.stack([
            flp[i, fi] + rlp[i, ri]
            for fi, ri in pairs
        ])
        probs = torch.softmax(scores, dim=0)
        order = torch.argsort(probs, descending=True)

        best = int(order[0])
        fi, ri = pairs[best]
        p1 = float(probs[best].detach().cpu())
        p2 = (
            float(probs[int(order[1])].detach().cpu())
            if len(order) > 1 else 0.0
        )

        pred_fields.append(fi)
        pred_roles.append(ri)
        confidence.append(p1)
        margin.append(p1 - p2)

    return {
        "field_ids": np.asarray(pred_fields),
        "role_ids": np.asarray(pred_roles),
        "confidence": np.asarray(confidence),
        "margin": np.asarray(margin),
    }

for dim in ["power", "throughput", "unknown"]:
    print("\n", dim)
    print([
        (ID_TO_FIELD[f], ID_TO_ROLE[r])
        for f, r in VALID_PAIR_CACHE[dim]
    ])


## 10 — Loss, class weights ROLE, AdamW, scheduler et AMP

Les FIELD labels V3 sont parfaitement équilibrés, donc la FIELD loss reste standard.

Les ROLE labels sont naturellement déséquilibrés. On utilise des poids `1/sqrt(freq)` normalisés pour aider les rôles rares **sans surcorriger** la distribution.


In [ ]:
# ---------------------------------------------------------
# Losses
# ---------------------------------------------------------
field_loss_fn = nn.CrossEntropyLoss()

role_counts = np.bincount(
    [ROLE_TO_ID[r["role"]] for r in train_records],
    minlength=NUM_ROLES,
).astype(np.float64)

if USE_ROLE_CLASS_WEIGHTS:
    role_weights_np = 1.0 / np.sqrt(np.maximum(role_counts, 1.0))
    role_weights_np = role_weights_np / role_weights_np.mean()
else:
    role_weights_np = np.ones(NUM_ROLES, dtype=np.float64)

role_weights = torch.tensor(
    role_weights_np,
    dtype=torch.float32,
    device=DEVICE,
)

role_loss_fn = nn.CrossEntropyLoss(weight=role_weights)

print("ROLE counts / weights:")
for i, role in enumerate(ROLE_LABELS):
    print(f"{role:18s} count={int(role_counts[i]):6d} weight={role_weights_np[i]:.4f}")

# ---------------------------------------------------------
# Optimizer: lower LR for pretrained encoder, higher for new heads.
# No weight decay on bias / LayerNorm parameters.
# ---------------------------------------------------------
def split_decay_params(module):
    decay, no_decay = [], []
    for name, param in module.named_parameters():
        if not param.requires_grad:
            continue
        if name.endswith("bias") or "LayerNorm.weight" in name or "layer_norm.weight" in name:
            no_decay.append(param)
        else:
            decay.append(param)
    return decay, no_decay

enc_decay, enc_no_decay = split_decay_params(model.encoder)
head_decay = list(model.field_head.parameters()) + list(model.role_head.parameters())

optimizer = AdamW(
    [
        {"params": enc_decay, "lr": ENCODER_LR, "weight_decay": WEIGHT_DECAY},
        {"params": enc_no_decay, "lr": ENCODER_LR, "weight_decay": 0.0},
        {"params": head_decay, "lr": HEAD_LR, "weight_decay": WEIGHT_DECAY},
    ]
)

steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
total_steps = steps_per_epoch * MAX_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

try:
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
except TypeError:
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

def autocast_context():
    return torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
        enabled=USE_AMP,
    )

print("Batches/epoch:", len(train_loader))
print("Optimizer steps/epoch:", steps_per_epoch)
print("Total scheduled optimizer steps:", total_steps)
print("Warmup steps:", warmup_steps)


## 11 — Fonctions d'entraînement et d'évaluation

Le scheduler n'avance que si le `optimizer.step()` AMP a réellement été exécuté.

L'évaluation conserve les logits et les métadonnées afin de produire ensuite :
- métriques globales ;
- métriques par langue ;
- métriques par stress family ;
- évaluation spécifique des groupes `alternative_pair`.


In [ ]:
def train_one_epoch(model, loader, epoch):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    total = field_total = role_total = 0.0
    optimizer_updates = amp_skipped_updates = 0

    progress = tqdm(loader, desc=f"TRAIN epoch {epoch}", leave=False)

    for bi, b in enumerate(progress):
        ids = b["input_ids"].to(DEVICE, non_blocking=True)
        mask = b["attention_mask"].to(DEVICE, non_blocking=True)
        fy = b["field_labels"].to(DEVICE, non_blocking=True)
        ry = b["role_labels"].to(DEVICE, non_blocking=True)

        with autocast_context():
            fl, rl = model(ids, mask)
            lf = field_loss_fn(fl, fy)
            lr = role_loss_fn(rl, ry)
            loss = FIELD_LOSS_WEIGHT * lf + ROLE_LOSS_WEIGHT * lr
            scaled_loss = loss / GRAD_ACCUM_STEPS

        scaler.scale(scaled_loss).backward()

        do_step = (
            (bi + 1) % GRAD_ACCUM_STEPS == 0
            or (bi + 1) == len(loader)
        )

        if do_step:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)

            scale_before = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            scale_after = scaler.get_scale()

            optimizer_step_executed = scale_after >= scale_before

            if optimizer_step_executed:
                scheduler.step()
                optimizer_updates += 1
            else:
                amp_skipped_updates += 1

            optimizer.zero_grad(set_to_none=True)

        total += float(loss.detach().cpu())
        field_total += float(lf.detach().cpu())
        role_total += float(lr.detach().cpu())

        if (bi + 1) % 200 == 0:
            progress.set_postfix(
                loss=f"{total/(bi+1):.4f}",
                lr=f"{scheduler.get_last_lr()[0]:.2e}",
            )

    n = len(loader)
    return {
        "loss": total / n,
        "field_loss": field_total / n,
        "role_loss": role_total / n,
        "optimizer_updates": optimizer_updates,
        "amp_skipped_updates": amp_skipped_updates,
        "last_lr": float(scheduler.get_last_lr()[0]),
    }


@torch.no_grad()
def collect_predictions(model, loader, desc="EVAL"):
    model.eval()

    FL, RL = [], []
    TF, TR = [], []
    dims, sids, gids, langs, sources = [], [], [], [], []

    loss_total = 0.0

    for b in tqdm(loader, desc=desc, leave=False):
        ids = b["input_ids"].to(DEVICE, non_blocking=True)
        mask = b["attention_mask"].to(DEVICE, non_blocking=True)
        fy = b["field_labels"].to(DEVICE, non_blocking=True)
        ry = b["role_labels"].to(DEVICE, non_blocking=True)

        with autocast_context():
            fl, rl = model(ids, mask)
            lf = field_loss_fn(fl, fy)
            lr = role_loss_fn(rl, ry)
            loss = FIELD_LOSS_WEIGHT * lf + ROLE_LOSS_WEIGHT * lr

        loss_total += float(loss.detach().cpu())

        FL.append(fl.float().cpu())
        RL.append(rl.float().cpu())
        TF.extend(fy.cpu().numpy().tolist())
        TR.extend(ry.cpu().numpy().tolist())

        dims.extend(b["dimensions"])
        sids.extend(b["sample_ids"])
        gids.extend(b["group_ids"])
        langs.extend(b["languages"])
        sources.extend(b["sources"])

    FL = torch.cat(FL)
    RL = torch.cat(RL)
    TF = np.asarray(TF)
    TR = np.asarray(TR)

    dec = decode_valid_pairs(FL, RL, dims)
    PF = dec["field_ids"]
    PR = dec["role_ids"]

    true_pairs = [f"{f}::{r}" for f, r in zip(TF, TR)]
    pred_pairs = [f"{f}::{r}" for f, r in zip(PF, PR)]

    umask = TF == UNRESOLVED_FIELD_ID
    unresolved_recall = (
        float(np.mean(PF[umask] == UNRESOLVED_FIELD_ID))
        if umask.any() else float("nan")
    )

    metrics = {
        "loss": loss_total / len(loader),
        "field_accuracy": accuracy_score(TF, PF),
        "field_macro_f1": f1_score(TF, PF, average="macro", zero_division=0),
        "role_accuracy": accuracy_score(TR, PR),
        "role_macro_f1": f1_score(TR, PR, average="macro", zero_division=0),
        "pair_accuracy": accuracy_score(true_pairs, pred_pairs),
        "pair_macro_f1": f1_score(true_pairs, pred_pairs, average="macro", zero_division=0),
        "unresolved_recall": unresolved_recall,
    }

    return {
        "metrics": metrics,
        "field_logits": FL,
        "role_logits": RL,
        "true_fields": TF,
        "true_roles": TR,
        "pred_fields": PF,
        "pred_roles": PR,
        "confidence": dec["confidence"],
        "margin": dec["margin"],
        "dimensions": np.asarray(dims),
        "sample_ids": np.asarray(sids),
        "group_ids": np.asarray(gids),
        "languages": np.asarray(langs),
        "sources": np.asarray(sources),
    }


## 12 — Fine-tuning end-to-end

Checkpoint sélectionné sur **VAL pair macro-F1** seulement.

Le `TEST` n'est pas lu ici. Early stopping évite de forcer toutes les epochs si le modèle a déjà convergé.


In [ ]:
history = []
best_val_score = -1.0
best_epoch = None
no_improve = 0

BEST_STATE_PATH = BEST_DIR / "semantic_linker_v3_best_state.pt"

training_started = time.time()

for epoch in range(1, MAX_EPOCHS + 1):
    print("\n" + "=" * 78)
    print(f"EPOCH {epoch}/{MAX_EPOCHS}")
    print("=" * 78)

    epoch_started = time.time()

    tm = train_one_epoch(model, train_loader, epoch)
    vo = collect_predictions(model, val_loader, desc=f"VAL epoch {epoch}")
    vm = vo["metrics"]

    row = {
        "epoch": epoch,
        "train_loss": tm["loss"],
        "train_field_loss": tm["field_loss"],
        "train_role_loss": tm["role_loss"],
        "optimizer_updates": tm["optimizer_updates"],
        "amp_skipped_updates": tm["amp_skipped_updates"],
        "last_lr": tm["last_lr"],
        "val_loss": vm["loss"],
        "val_field_accuracy": vm["field_accuracy"],
        "val_field_macro_f1": vm["field_macro_f1"],
        "val_role_accuracy": vm["role_accuracy"],
        "val_role_macro_f1": vm["role_macro_f1"],
        "val_pair_accuracy": vm["pair_accuracy"],
        "val_pair_macro_f1": vm["pair_macro_f1"],
        "val_unresolved_recall": vm["unresolved_recall"],
        "epoch_minutes": (time.time() - epoch_started) / 60.0,
    }

    history.append(row)

    print(
        f"train_loss={row['train_loss']:.4f} | "
        f"val_loss={row['val_loss']:.4f}"
    )
    print(
        f"FIELD F1={row['val_field_macro_f1']:.4f} | "
        f"ROLE F1={row['val_role_macro_f1']:.4f} | "
        f"PAIR F1={row['val_pair_macro_f1']:.4f} | "
        f"UNRES recall={row['val_unresolved_recall']:.4f}"
    )
    print(
        f"optimizer_updates={row['optimizer_updates']} | "
        f"AMP skipped={row['amp_skipped_updates']} | "
        f"epoch={row['epoch_minutes']:.1f} min"
    )

    score = row["val_pair_macro_f1"]

    if score > best_val_score + 1e-6:
        best_val_score = score
        best_epoch = epoch
        no_improve = 0

        torch.save(model.state_dict(), BEST_STATE_PATH)
        print("✅ New best checkpoint")
    else:
        no_improve += 1
        print("No improvement:", no_improve, "/", EARLY_STOPPING_PATIENCE)

        if no_improve >= EARLY_STOPPING_PATIENCE:
            print("⏹ Early stopping")
            break

history_df = pd.DataFrame(history)
print("\nBest epoch:", best_epoch)
print("Best VAL pair macro-F1:", best_val_score)
print("Total training minutes:", (time.time() - training_started) / 60.0)

display(history_df)


## 13 — Courbes d'apprentissage


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history_df.epoch, history_df.train_loss, marker="o", label="train")
ax.plot(history_df.epoch, history_df.val_loss, marker="o", label="val")
ax.set_title("Semantic Linker V3 — Loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.legend()
ax.grid(alpha=.2)
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history_df.epoch, history_df.val_field_macro_f1, marker="o", label="FIELD")
ax.plot(history_df.epoch, history_df.val_role_macro_f1, marker="o", label="ROLE")
ax.plot(history_df.epoch, history_df.val_pair_macro_f1, marker="o", label="PAIR")
ax.set_ylim(0, 1.02)
ax.set_title("Semantic Linker V3 — Validation macro-F1")
ax.set_xlabel("Epoch")
ax.set_ylabel("Macro-F1")
ax.legend()
ax.grid(alpha=.2)
plt.show()


## 14 — Recharger le meilleur checkpoint et analyser VAL

Cette étape précède la calibration. Le TEST reste encore fermé.


In [ ]:
model.load_state_dict(torch.load(BEST_STATE_PATH, map_location=DEVICE))
model.to(DEVICE)
model.eval()

val_output = collect_predictions(model, val_loader, desc="VAL best checkpoint")
val_metrics = val_output["metrics"]

print("BEST VALIDATION METRICS")
for k, v in val_metrics.items():
    print(f"{k:24s}: {v:.6f}")

print("\nFIELD report:")
print(classification_report(
    val_output["true_fields"],
    val_output["pred_fields"],
    labels=list(range(NUM_FIELDS)),
    target_names=FIELD_LABELS,
    zero_division=0,
    digits=4,
))

print("\nROLE report:")
print(classification_report(
    val_output["true_roles"],
    val_output["pred_roles"],
    labels=list(range(NUM_ROLES)),
    target_names=ROLE_LABELS,
    zero_division=0,
    digits=4,
))


## 15 — Calibration sélective `confidence + margin` sur VAL

Objectif :
1. `accepted_precision ≥ 98 %`;
2. `unresolved_recall ≥ 99.5 %`;
3. parmi ces solutions, maximiser la coverage.

La grille combine des thresholds fixes et des quantiles réellement observés sur VAL, ce qui évite de supposer à l'avance que le nouveau modèle aura la même calibration que V1.


In [ ]:
def selective_arrays(output, confidence_threshold, margin_threshold):
    TF = output["true_fields"]
    TR = output["true_roles"]
    PF = output["pred_fields"].copy()
    PR = output["pred_roles"].copy()

    accepted = (
        (PF != UNRESOLVED_FIELD_ID)
        & (output["confidence"] >= confidence_threshold)
        & (output["margin"] >= margin_threshold)
    )

    FF = PF.copy()
    FR = PR.copy()

    FF[~accepted] = UNRESOLVED_FIELD_ID
    FR[~accepted] = UNSPECIFIED_ROLE_ID

    return accepted, FF, FR

def selective_metrics(output, confidence_threshold, margin_threshold):
    TF = output["true_fields"]
    TR = output["true_roles"]
    PF = output["pred_fields"]
    PR = output["pred_roles"]

    accepted, FF, FR = selective_arrays(
        output,
        confidence_threshold,
        margin_threshold,
    )

    correct = (PF == TF) & (PR == TR)
    accepted_count = int(accepted.sum())

    accepted_precision = (
        float(correct[accepted].mean())
        if accepted_count > 0 else np.nan
    )

    unresolved_mask = TF == UNRESOLVED_FIELD_ID
    unresolved_recall = (
        float(np.mean(FF[unresolved_mask] == UNRESOLVED_FIELD_ID))
        if unresolved_mask.any() else np.nan
    )

    return {
        "confidence_threshold": float(confidence_threshold),
        "margin_threshold": float(margin_threshold),
        "accepted_count": accepted_count,
        "accepted_precision": accepted_precision,
        "coverage": float(accepted.mean()),
        "unresolved_recall": unresolved_recall,
        "final_pair_accuracy": float(np.mean((FF == TF) & (FR == TR))),
        "final_field_macro_f1": float(
            f1_score(TF, FF, average="macro", zero_division=0)
        ),
    }

def candidate_thresholds(values, fixed_grid):
    qs = np.quantile(
        values,
        [0, .01, .02, .05, .10, .15, .20, .30, .40, .50,
         .60, .70, .80, .85, .90, .92, .94, .95, .96, .97,
         .98, .99, .995, 1.0]
    )
    merged = np.concatenate([np.asarray(fixed_grid), qs])
    merged = np.clip(merged, 0.0, 0.999999)
    return np.unique(np.round(merged, 6))

confidence_values = candidate_thresholds(
    val_output["confidence"],
    np.arange(0.50, 0.995, 0.01),
)

margin_values = candidate_thresholds(
    val_output["margin"],
    np.arange(0.0, 0.81, 0.025),
)

print("Confidence candidates:", len(confidence_values))
print("Margin candidates:", len(margin_values))

rows = []

for c in tqdm(confidence_values, desc="Threshold calibration"):
    for m in margin_values:
        rows.append(selective_metrics(val_output, c, m))

threshold_df = pd.DataFrame(rows)
valid = threshold_df[threshold_df["accepted_count"] > 0].copy()

eligible = valid[
    (valid["accepted_precision"] >= TARGET_ACCEPTED_PRECISION)
    & (valid["unresolved_recall"] >= MIN_UNRESOLVED_RECALL)
].copy()

if len(eligible) > 0:
    selected = (
        eligible
        .sort_values(
            ["coverage", "accepted_precision", "final_pair_accuracy"],
            ascending=False,
        )
        .iloc[0]
    )
    threshold_validated = True
    selection_reason = (
        f"accepted_precision >= {TARGET_ACCEPTED_PRECISION:.2%}; "
        f"unresolved_recall >= {MIN_UNRESOLVED_RECALL:.2%}; "
        "then maximum coverage"
    )
else:
    selected = (
        valid
        .sort_values(
            ["accepted_precision", "unresolved_recall", "coverage"],
            ascending=False,
        )
        .iloc[0]
    )
    threshold_validated = False
    selection_reason = "No threshold satisfies both safety targets"

CONFIDENCE_THRESHOLD = float(selected["confidence_threshold"])
MARGIN_THRESHOLD = float(selected["margin_threshold"])

print("\n" + "=" * 78)
print("✅ THRESHOLD VALIDATED" if threshold_validated else "⚠️ SAFETY TARGET NOT REACHED")
print("=" * 78)
for key in [
    "confidence_threshold",
    "margin_threshold",
    "accepted_precision",
    "coverage",
    "accepted_count",
    "unresolved_recall",
    "final_pair_accuracy",
    "final_field_macro_f1",
]:
    print(f"{key:26s}: {selected[key]}")
print("Selection reason:", selection_reason)

display(
    valid.sort_values(
        ["accepted_precision", "coverage"],
        ascending=False,
    ).head(30)
)


## 16 — Risk–Coverage sur VAL

La courbe montre le compromis : plus la politique accepte de cas, plus le risque peut augmenter.

Le point choisi est annoté explicitement avec `confidence threshold` et `margin threshold`.


In [ ]:
# Keep, for each rounded coverage bin, the lowest-risk configuration.
risk_df = valid.copy()
risk_df["risk"] = 1.0 - risk_df["accepted_precision"]
risk_df["coverage_bin"] = (risk_df["coverage"] * 200).round() / 200

frontier = (
    risk_df
    .sort_values(["coverage_bin", "risk"])
    .groupby("coverage_bin", as_index=False)
    .first()
    .sort_values("coverage")
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(frontier["coverage"], frontier["risk"], marker="o", markersize=3)
ax.scatter(
    [selected["coverage"]],
    [1.0 - selected["accepted_precision"]],
    s=100,
    zorder=5,
)
ax.annotate(
    f"chosen\nconf={CONFIDENCE_THRESHOLD:.3f}\nmargin={MARGIN_THRESHOLD:.3f}",
    (selected["coverage"], 1.0 - selected["accepted_precision"]),
    xytext=(10, 15),
    textcoords="offset points",
)
ax.set_xlabel("Coverage (fraction of examples automatically accepted)")
ax.set_ylabel("Risk = 1 - accepted precision")
ax.set_title("VAL — Selective Risk–Coverage")
ax.grid(alpha=.2)
plt.show()


# 17 — TEST final : première et unique ouverture

À partir de cette cellule :
- modèle figé ;
- hyperparamètres figés ;
- thresholds figés.

**Ne retune rien en regardant ces résultats.**  
Si une future V4 est créée à partir des erreurs TEST, elle devra utiliser un nouveau TEST indépendant.


In [ ]:
test_output = collect_predictions(model, test_loader, desc="FINAL TEST")
test_metrics = test_output["metrics"]

test_selective = selective_metrics(
    test_output,
    CONFIDENCE_THRESHOLD,
    MARGIN_THRESHOLD,
)

print("TEST — RAW CONSTRAINED")
for k, v in test_metrics.items():
    print(f"{k:24s}: {v:.6f}")

print("\nTEST — SELECTIVE")
for k, v in test_selective.items():
    print(f"{k:26s}: {v}")

print("\nFIELD report:")
print(classification_report(
    test_output["true_fields"],
    test_output["pred_fields"],
    labels=list(range(NUM_FIELDS)),
    target_names=FIELD_LABELS,
    zero_division=0,
    digits=4,
))

print("\nROLE report:")
print(classification_report(
    test_output["true_roles"],
    test_output["pred_roles"],
    labels=list(range(NUM_ROLES)),
    target_names=ROLE_LABELS,
    zero_division=0,
    digits=4,
))


## 18 — Confusion matrices TEST


In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
ConfusionMatrixDisplay.from_predictions(
    test_output["true_fields"],
    test_output["pred_fields"],
    labels=list(range(NUM_FIELDS)),
    display_labels=FIELD_LABELS,
    xticks_rotation=90,
    cmap="Blues",
    ax=ax,
    colorbar=False,
)
ax.set_title("TEST V3 — FIELD confusion matrix")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(11, 9))
ConfusionMatrixDisplay.from_predictions(
    test_output["true_roles"],
    test_output["pred_roles"],
    labels=list(range(NUM_ROLES)),
    display_labels=ROLE_LABELS,
    xticks_rotation=90,
    cmap="Blues",
    ax=ax,
    colorbar=False,
)
ax.set_title("TEST V3 — ROLE confusion matrix")
plt.tight_layout()
plt.show()


## 19 — Métriques TEST par langue, dimension et famille de stress

Cette section est indispensable pour V3 : une moyenne globale élevée ne suffit pas.

On veut savoir précisément si le modèle résiste à :
- `semantic_anchor_typo`;
- `noisy_mixed_typos`;
- `mixed_code_switch`;
- `hpc_hard_negative`;
- `multi_quantity_collision`;
- `alternative_pair`;
- `quantity_words`;
- etc.


In [ ]:
test_accepted, test_final_fields, test_final_roles = selective_arrays(
    test_output,
    CONFIDENCE_THRESHOLD,
    MARGIN_THRESHOLD,
)

def subset_metrics(output, indices):
    idx = np.asarray(indices, dtype=int)

    TF = output["true_fields"][idx]
    TR = output["true_roles"][idx]
    PF = output["pred_fields"][idx]
    PR = output["pred_roles"][idx]

    accepted = test_accepted[idx]
    FF = test_final_fields[idx]
    FR = test_final_roles[idx]

    correct_pair = (PF == TF) & (PR == TR)
    accepted_count = int(accepted.sum())

    unresolved_mask = TF == UNRESOLVED_FIELD_ID

    return {
        "n": int(len(idx)),
        "raw_field_macro_f1": float(
            f1_score(TF, PF, average="macro", zero_division=0)
        ),
        "raw_pair_accuracy": float(np.mean(correct_pair)),
        "selective_coverage": float(accepted.mean()),
        "selective_precision": (
            float(correct_pair[accepted].mean())
            if accepted_count else np.nan
        ),
        "final_pair_accuracy": float(np.mean((FF == TF) & (FR == TR))),
        "unresolved_recall": (
            float(np.mean(FF[unresolved_mask] == UNRESOLVED_FIELD_ID))
            if unresolved_mask.any() else np.nan
        ),
    }

def grouped_metrics(output, labels):
    rows = []
    labels = np.asarray(labels)
    for label in sorted(set(labels.tolist())):
        idx = np.where(labels == label)[0]
        row = {"group": label}
        row.update(subset_metrics(output, idx))
        rows.append(row)
    return pd.DataFrame(rows)

language_metrics_df = grouped_metrics(
    test_output,
    test_output["languages"],
).rename(columns={"group": "language"})

dimension_metrics_df = grouped_metrics(
    test_output,
    test_output["dimensions"],
).rename(columns={"group": "dimension"})

test_stress = np.asarray([
    s.split("+")[-1] if "+" in s else s
    for s in test_output["sources"]
])

stress_metrics_df = grouped_metrics(
    test_output,
    test_stress,
).rename(columns={"group": "stress_family"})

print("BY LANGUAGE")
display(language_metrics_df)

print("BY DIMENSION")
display(dimension_metrics_df)

print("BY STRESS FAMILY")
display(
    stress_metrics_df.sort_values(
        ["final_pair_accuracy", "raw_pair_accuracy"],
        ascending=True,
    )
)


## 20 — Évaluation spéciale `alternative_pair`

Pour V3, **`X or Y` n'est pas une abstention du Semantic Linker**.

Exemple :
- `[Q]20 GB/s[/Q] or 30 GB/s` → `target_write_gbps`
- `20 GB/s or [Q]30 GB/s[/Q]` → `target_write_gbps`

Le guard downstream décidera ensuite quelle valeur conserver.

Cette cellule mesure donc la proportion de groupes où **les deux quantités** sont correctement classifiées, et où les deux passent éventuellement la politique sélective.


In [ ]:
alt_idx = np.where(test_stress == "alternative_pair")[0]
assert len(alt_idx) > 0

groups = defaultdict(list)
for i in alt_idx:
    groups[test_output["group_ids"][i]].append(int(i))

bad_group_sizes = {
    gid: len(indices)
    for gid, indices in groups.items()
    if len(indices) != 2
}
assert not bad_group_sizes, list(bad_group_sizes.items())[:10]

raw_group_success = []
selective_group_success = []
same_gold_field = []

for gid, indices in groups.items():
    idx = np.asarray(indices)
    TF = test_output["true_fields"][idx]
    TR = test_output["true_roles"][idx]
    PF = test_output["pred_fields"][idx]
    PR = test_output["pred_roles"][idx]

    same_gold_field.append(len(set(TF.tolist())) == 1)

    raw_group_success.append(
        bool(np.all((PF == TF) & (PR == TR)))
    )

    selective_group_success.append(
        bool(
            np.all(test_accepted[idx])
            and np.all((PF == TF) & (PR == TR))
        )
    )

alternative_pair_metrics = {
    "groups": len(groups),
    "samples": len(alt_idx),
    "all_groups_have_same_gold_field": bool(all(same_gold_field)),
    "raw_both_quantities_correct_rate": float(np.mean(raw_group_success)),
    "selective_both_quantities_accepted_and_correct_rate": float(
        np.mean(selective_group_success)
    ),
}

print(json.dumps(alternative_pair_metrics, indent=2))


## 21 — Diagnostics ciblés après TEST

Ces probes vérifient les faiblesses qui avaient motivé V3.

⚠️ **Ils ne font pas partie des métriques TEST indépendantes**, car leurs familles ont influencé la conception du nouveau dataset.

Ce qu'on veut voir :
- `clents`, `clints` → `client_count`;
- `noeuds compute` → `client_count`;
- un faux ami HPC (`CPU cores`) → idéalement `__UNRESOLVED__`;
- les deux côtés de `20 or 30 GB/s` → même field `target_write_gbps`.


In [ ]:
@torch.no_grad()
def predict_one(marked_text, dimension):
    if dimension not in ALLOWED_FIELDS_BY_DIMENSION:
        raise ValueError(dimension)

    enc = tokenizer(
        marked_text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
    )

    model.eval()
    with autocast_context():
        fl, rl = model(
            enc["input_ids"].to(DEVICE),
            enc["attention_mask"].to(DEVICE),
        )

    dec = decode_valid_pairs(
        fl.float().cpu(),
        rl.float().cpu(),
        [dimension],
    )

    fi = int(dec["field_ids"][0])
    ri = int(dec["role_ids"][0])
    conf = float(dec["confidence"][0])
    margin = float(dec["margin"][0])

    accepted = (
        fi != UNRESOLVED_FIELD_ID
        and conf >= CONFIDENCE_THRESHOLD
        and margin >= MARGIN_THRESHOLD
    )

    return {
        "raw_best_field": ID_TO_FIELD[fi],
        "raw_best_role": ID_TO_ROLE[ri],
        "confidence": conf,
        "margin": margin,
        "accepted": accepted,
        "final_field": ID_TO_FIELD[fi] if accepted else "__UNRESOLVED__",
        "final_role": ID_TO_ROLE[ri] if accepted else "unspecified",
    }

diagnostic_examples = [
    ("There are [Q]200[/Q] clents.", "unknown"),
    ("Il y aura [Q]250[/Q] clints sur le système.", "unknown"),
    ("Approximately [Q]260[/Q] noeuds compute will mount Lustre.", "unknown"),
    ("The cluster has [Q]200[/Q] CPU cores, not Lustre clients.", "unknown"),
    ("Write throughput could be [Q]20 GB/s[/Q] or 30 GB/s.", "throughput"),
    ("Write throughput could be 20 GB/s or [Q]30 GB/s[/Q].", "throughput"),
]

for text, dim in diagnostic_examples:
    print("\nTEXT:", text)
    print("DIM :", dim)
    print(json.dumps(predict_one(text, dim), indent=2, ensure_ascii=False))


## 22 — Construire les prédictions TEST traçables

On exporte une ligne par exemple avec :
- gold FIELD/ROLE ;
- raw FIELD/ROLE ;
- confidence/margin ;
- décision sélective ;
- langue ;
- stress family ;
- group_id.

Cela permettra d'analyser V3 localement sans relancer Kaggle.


In [ ]:
prediction_rows = []

for i in range(len(test_output["sample_ids"])):
    prediction_rows.append({
        "sample_id": str(test_output["sample_ids"][i]),
        "group_id": str(test_output["group_ids"][i]),
        "language": str(test_output["languages"][i]),
        "dimension": str(test_output["dimensions"][i]),
        "stress_family": str(test_stress[i]),
        "true_field": ID_TO_FIELD[int(test_output["true_fields"][i])],
        "true_role": ID_TO_ROLE[int(test_output["true_roles"][i])],
        "raw_pred_field": ID_TO_FIELD[int(test_output["pred_fields"][i])],
        "raw_pred_role": ID_TO_ROLE[int(test_output["pred_roles"][i])],
        "confidence": float(test_output["confidence"][i]),
        "margin": float(test_output["margin"][i]),
        "accepted": bool(test_accepted[i]),
        "final_field": ID_TO_FIELD[int(test_final_fields[i])],
        "final_role": ID_TO_ROLE[int(test_final_roles[i])],
    })

test_predictions_df = pd.DataFrame(prediction_rows)
display(test_predictions_df.head())
print("rows:", len(test_predictions_df))


## 23 — Export complet compatible avec `SemanticLinkerRuntime`

Le ZIP final n'inclut **pas** un deuxième state-dict complet redondant.  
`encoder/ + classifier_heads.pt + tokenizer/` suffisent pour reconstruire exactement le runtime, ce qui réduit fortement la taille de l'artefact.

Le checkpoint d'entraînement complet reste temporairement dans `/kaggle/working/semantic_linker_xlmr_v3/best_checkpoint`.


In [ ]:
if EXPORT_DIR.exists():
    shutil.rmtree(EXPORT_DIR)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# Runtime artifacts
# ---------------------------------------------------------
model.encoder.save_pretrained(
    EXPORT_DIR / "encoder",
    safe_serialization=True,
)
tokenizer.save_pretrained(EXPORT_DIR / "tokenizer")

torch.save(
    {
        "field_head": model.field_head.state_dict(),
        "role_head": model.role_head.state_dict(),
        "dropout": DROPOUT,
        "hidden_size": model.encoder.config.hidden_size,
        "num_fields": NUM_FIELDS,
        "num_roles": NUM_ROLES,
    },
    EXPORT_DIR / "classifier_heads.pt",
)

(EXPORT_DIR / "labels.json").write_text(
    json.dumps(
        {
            "field_labels": FIELD_LABELS,
            "role_labels": ROLE_LABELS,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

(EXPORT_DIR / "compatibility.json").write_text(
    json.dumps(
        {
            "allowed_fields_by_dimension": {
                k: sorted(v)
                for k, v in ALLOWED_FIELDS_BY_DIMENSION.items()
            },
            "allowed_roles_by_field": {
                k: sorted(v)
                for k, v in ALLOWED_ROLES_BY_FIELD.items()
            },
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

(EXPORT_DIR / "thresholds.json").write_text(
    json.dumps(
        {
            "confidence_threshold": CONFIDENCE_THRESHOLD,
            "margin_threshold": MARGIN_THRESHOLD,
            "selection_reason": selection_reason,
            "threshold_validated": threshold_validated,
            "target_accepted_precision": TARGET_ACCEPTED_PRECISION,
            "minimum_unresolved_recall": MIN_UNRESOLVED_RECALL,
        },
        indent=2,
    ),
    encoding="utf-8",
)

training_config = {
    "model_source": MODEL_SOURCE,
    "dataset_version": "semantic_linker_v3_global_stress",
    "train_file": TRAIN_FILENAME,
    "val_file": VAL_FILENAME,
    "test_file": TEST_FILENAME,
    "train_size": len(train_records),
    "val_size": len(val_records),
    "test_size": len(test_records),
    "seed": SEED,
    "max_length": MAX_LENGTH,
    "batch_size": BATCH_SIZE,
    "gradient_accumulation_steps": GRAD_ACCUM_STEPS,
    "effective_batch_size": EFFECTIVE_BATCH_SIZE,
    "max_epochs": MAX_EPOCHS,
    "best_epoch": best_epoch,
    "encoder_lr": ENCODER_LR,
    "head_lr": HEAD_LR,
    "weight_decay": WEIGHT_DECAY,
    "warmup_ratio": WARMUP_RATIO,
    "dropout": DROPOUT,
    "amp": USE_AMP,
    "gradient_checkpointing": USE_GRADIENT_CHECKPOINTING,
    "role_class_weights_enabled": USE_ROLE_CLASS_WEIGHTS,
    "role_class_weights": {
        ROLE_LABELS[i]: float(role_weights_np[i])
        for i in range(NUM_ROLES)
    },
    "special_tokens": SPECIAL_TOKENS,
}

(EXPORT_DIR / "training_config.json").write_text(
    json.dumps(training_config, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

# ---------------------------------------------------------
# Metrics / diagnostics
# ---------------------------------------------------------
(EXPORT_DIR / "validation_metrics.json").write_text(
    json.dumps(
        {
            "raw_constrained": {
                k: float(v) for k, v in val_metrics.items()
            },
            "selected_threshold": {
                k: (
                    int(v) if k == "accepted_count"
                    else bool(v) if isinstance(v, (bool, np.bool_))
                    else float(v)
                )
                for k, v in selected.to_dict().items()
            },
        },
        indent=2,
    ),
    encoding="utf-8",
)

(EXPORT_DIR / "test_metrics.json").write_text(
    json.dumps(
        {
            "raw_constrained": {
                k: float(v) for k, v in test_metrics.items()
            },
            "selective": {
                k: (
                    int(v) if k == "accepted_count"
                    else float(v)
                )
                for k, v in test_selective.items()
            },
        },
        indent=2,
    ),
    encoding="utf-8",
)

(EXPORT_DIR / "alternative_pair_metrics.json").write_text(
    json.dumps(alternative_pair_metrics, indent=2),
    encoding="utf-8",
)

(EXPORT_DIR / "training_history.json").write_text(
    json.dumps(history, indent=2),
    encoding="utf-8",
)

threshold_df.to_csv(
    EXPORT_DIR / "validation_threshold_grid.csv",
    index=False,
)
language_metrics_df.to_csv(
    EXPORT_DIR / "test_metrics_by_language.csv",
    index=False,
)
dimension_metrics_df.to_csv(
    EXPORT_DIR / "test_metrics_by_dimension.csv",
    index=False,
)
stress_metrics_df.to_csv(
    EXPORT_DIR / "test_metrics_by_stress_family.csv",
    index=False,
)
test_predictions_df.to_json(
    EXPORT_DIR / "test_predictions.jsonl",
    orient="records",
    lines=True,
    force_ascii=False,
)

# Risk-coverage plot for report.
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(frontier["coverage"], frontier["risk"], marker="o", markersize=3)
ax.scatter(
    [selected["coverage"]],
    [1.0 - selected["accepted_precision"]],
    s=100,
    zorder=5,
)
ax.annotate(
    f"chosen\nconf={CONFIDENCE_THRESHOLD:.3f}\nmargin={MARGIN_THRESHOLD:.3f}",
    (selected["coverage"], 1.0 - selected["accepted_precision"]),
    xytext=(10, 15),
    textcoords="offset points",
)
ax.set_xlabel("Coverage")
ax.set_ylabel("Risk = 1 - accepted precision")
ax.set_title("Semantic Linker V3 — VAL Risk–Coverage")
ax.grid(alpha=.2)
fig.tight_layout()
fig.savefig(EXPORT_DIR / "validation_risk_coverage.png", dpi=180)
plt.close(fig)

# ---------------------------------------------------------
# ZIP
# ---------------------------------------------------------
ZIP_BASE = "/kaggle/working/semantic_linker_xlmr_base_v3_final"
ZIP_PATH = shutil.make_archive(
    ZIP_BASE,
    "zip",
    root_dir=EXPORT_DIR,
)

print("✅ Export directory:", EXPORT_DIR)
print("✅ ZIP:", ZIP_PATH)
print("\nExported files:")
for p in sorted(EXPORT_DIR.rglob("*")):
    if p.is_file():
        print(" -", p.relative_to(EXPORT_DIR))


## 24 — Smoke test de l'artefact exporté

On recharge **depuis les fichiers exportés**, pas depuis les objets Python d'entraînement.  
Cela vérifie que le ZIP contient réellement tout ce dont le runtime a besoin.


In [ ]:
# Lightweight contract checks.
required_export_files = [
    EXPORT_DIR / "classifier_heads.pt",
    EXPORT_DIR / "labels.json",
    EXPORT_DIR / "compatibility.json",
    EXPORT_DIR / "thresholds.json",
    EXPORT_DIR / "training_config.json",
    EXPORT_DIR / "encoder" / "config.json",
]

for p in required_export_files:
    assert p.exists(), p

exported_labels = json.loads(
    (EXPORT_DIR / "labels.json").read_text(encoding="utf-8")
)
exported_thresholds = json.loads(
    (EXPORT_DIR / "thresholds.json").read_text(encoding="utf-8")
)

assert exported_labels["field_labels"] == FIELD_LABELS
assert exported_labels["role_labels"] == ROLE_LABELS
assert 0.0 <= exported_thresholds["confidence_threshold"] <= 1.0
assert 0.0 <= exported_thresholds["margin_threshold"] <= 1.0

heads = torch.load(
    EXPORT_DIR / "classifier_heads.pt",
    map_location="cpu",
)
assert heads["num_fields"] == NUM_FIELDS
assert heads["num_roles"] == NUM_ROLES
assert heads["hidden_size"] == model.encoder.config.hidden_size

print("✅ Export contract valid")
print("confidence threshold:", exported_thresholds["confidence_threshold"])
print("margin threshold:", exported_thresholds["margin_threshold"])


## 25 — Résumé final V3

Cette cellule affiche les chiffres à copier dans le rapport avant le nouveau benchmark End-to-End du pipeline.


In [ ]:
summary = {
    "best_epoch": int(best_epoch),
    "val_pair_macro_f1": float(val_metrics["pair_macro_f1"]),
    "test_field_macro_f1": float(test_metrics["field_macro_f1"]),
    "test_role_macro_f1": float(test_metrics["role_macro_f1"]),
    "test_pair_macro_f1": float(test_metrics["pair_macro_f1"]),
    "test_unresolved_recall_raw": float(test_metrics["unresolved_recall"]),
    "confidence_threshold": float(CONFIDENCE_THRESHOLD),
    "margin_threshold": float(MARGIN_THRESHOLD),
    "test_selective_precision": float(test_selective["accepted_precision"]),
    "test_selective_coverage": float(test_selective["coverage"]),
    "test_selective_final_pair_accuracy": float(test_selective["final_pair_accuracy"]),
    "alternative_pair_raw_both_correct": float(
        alternative_pair_metrics["raw_both_quantities_correct_rate"]
    ),
    "alternative_pair_selective_both_correct": float(
        alternative_pair_metrics[
            "selective_both_quantities_accepted_and_correct_rate"
        ]
    ),
    "zip_path": ZIP_PATH,
}

print(json.dumps(summary, indent=2))


# Fichier final à récupérer

Dans **Kaggle → Output**, récupérer :

```text
semantic_linker_xlmr_base_v3_final.zip
```

Puis localement :

1. garder les artefacts V1 comme baseline ;
2. extraire V3 dans un nouveau dossier d'artefacts ;
3. faire pointer `SemanticLinkerRuntime` vers V3 ;
4. lancer les smoke/regression tests ;
5. relancer un **nouveau End-to-End holdout indépendant** avant de déclarer V3 meilleure que V1.

Ne remplace pas définitivement V1 avant cette comparaison.


In [ ]:
from IPython.display import FileLink, display

zip_size_mb = os.path.getsize(ZIP_PATH) / (1024 ** 2)

print("=" * 78)
print("✅ SEMANTIC LINKER V3 READY")
print("=" * 78)
print("ZIP :", ZIP_PATH)
print(f"Size: {zip_size_mb:.2f} MB")
print("\nDownload:")
display(FileLink(ZIP_PATH))
